In [1]:
import torch
from torch import nn

### Exercise 1

What kinds of problems will occur if you change MySequential to store modules in a Python list?

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.modules_list = []
        for module in args:
            self.modules_list.append(module)

    def forward(self, X):
        for module in self.modules_list:
            X = module(X)
        return X

If `MySequential` stores modules in a plain Python list, the following problems can occur:

* submodules are not registered
* parameters are missing from `model.parameters()`
* optimizer will not update them
* `state_dict()` may not include them
* `.to(device)` may not move them
* printing and inspecting the model becomes unreliable

The forward computation may still appear to work, which makes the bug subtle.

### Exercise 2

Implement a module that takes two modules as an argument, say net1 and net2 and returns the concatenated output of both networks in the forward propagation. This is also called a parallel module.

In [2]:
class ParallelModule(nn.Module):
    def __init__(self, net1, net2):
        super().__init__()
        self.net1 = net1
        self.net2 = net2

    def forward(self, X):
        Y1 = self.net1(X)
        Y2 = self.net2(X)
        return torch.cat((Y1, Y2), dim=1)
    

net1 = nn.Sequential(
    nn.LazyLinear(8),
    nn.ReLU()
)

net2 = nn.Sequential(
    nn.LazyLinear(12),
    nn.ReLU()
)

net = ParallelModule(net1, net2)

X = torch.rand(2, 20)
Y = net(X)

print(Y.shape)

torch.Size([2, 20])


/Users/zouminghao/Desktop/d2l-notes-exercises/venv/lib/python3.11/site-packages/torch/nn/modules/lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


To implement a parallel module, we define a custom subclass of `nn.Module`. In the constructor, we store the two given modules as instance variables so that PyTorch can properly register their parameters. In the forward propagation method, we feed the same input to both subnetworks, collect their outputs, and concatenate them along the feature dimension using `torch.cat`.

### Exercise 3

Assume that you want to concatenate multiple instances of the same network. Implement a factory function that generates multiple instances of the same module and build a larger network from it.

In [3]:
class MultiParallel(nn.Module):
    def __init__(self, factory_fn, num_nets):
        super().__init__()
        self.nets = nn.ModuleList([
            factory_fn() for _ in range(num_nets)
        ])

    def forward(self, X):
        outputs = [net(X) for net in self.nets]
        return torch.cat(outputs, dim=1)

def make_net():
    return nn.Sequential(
        nn.LazyLinear(8),
        nn.ReLU()
    )

net = MultiParallel(make_net, num_nets=3)

X = torch.rand(2, 20)
Y = net(X)

print(Y.shape)

torch.Size([2, 24])


Think of this as:

* each subnet = “expert”
* each expert sees the same input
* each produces features
* concatenation = combining all experts

This is closely related to:

* ensemble methods
* multi-branch neural networks
* feature expansion